# Przykładowy projekt: sprzedaż kawiarni

**Odbiorca:** osoba prowadząca kawiarnię. **Zakres:** styczeń–marzec 2026, dane syntetyczne.

Pytania: (1) Jaki jest udział kategorii w przychodzie? (2) Jak rozkłada się dzienny przychód? (3) Jak zmienia się miesięczny przychód i średni przychód na obserwowany dzień?

Jedna pozycja może zawierać kilka sztuk. Przychód nie jest zyskiem. W tej demonstracji używamy przygotowanej kopii, aby skupić się na analizie; raport czyszczenia: 243 → 240 po odjęciu powtórzonych ID → 236 po odrzuceniu 3 brakujących ilości i 1 błędnej daty. Dwie kategorie odtworzono ze słownika produktów, ceny przekonwertowano z tekstu, nazwy przycięto.


In [ ]:
from pathlib import Path

# Notebook działa po uruchomieniu z katalogu kursu lub z jego podkatalogu.
KATALOG_KURSU = next(
    katalog for katalog in [Path.cwd(), *Path.cwd().parents]
    if (katalog / "dane" / "sprzedaz_kawiarni.csv").exists()
)
DANE = KATALOG_KURSU / "dane"
WYNIKI = KATALOG_KURSU / "wyniki"
WYNIKI.mkdir(exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt

sprzedaz = pd.read_csv(DANE / "sprzedaz_kawiarni_gotowa.csv", parse_dates=["data"])
sprzedaz.head()


In [ ]:
print("Rekordy:", len(sprzedaz))
print("Zakres:", sprzedaz["data"].min(), sprzedaz["data"].max())
print(sprzedaz.isna().sum())
print("Powtórzone ID:", sprzedaz.duplicated(subset="id_transakcji").sum())
sprzedaz["przychod"] = sprzedaz["liczba_sztuk"] * sprzedaz["cena_jednostkowa"]


## 1. Udział kategorii w przychodzie


In [ ]:
kategorie = sprzedaz.groupby("kategoria")["przychod"].sum().sort_values(ascending=False)
udzialy = kategorie / kategorie.sum() * 100
print(pd.DataFrame({"przychod_zl": kategorie, "udzial_proc": udzialy}).round(1))
ax = kategorie.plot.bar(rot=0, figsize=(6, 4))
ax.set(title="Przychód według kategorii", xlabel="Kategoria", ylabel="Przychód [zł]")
plt.tight_layout()
plt.show()
print(f"Największy udział ma kategoria {udzialy.index[0]}: {udzialy.iloc[0]:.1f}% przychodu.")


## 2. Rozkład dziennego przychodu

Sprawdzamy, czy wszystkie daty w okresie mają obserwacje. Brak obserwacji nie musiałby oznaczać zerowego przychodu.


In [ ]:
dziennie = sprzedaz.groupby("data")["przychod"].sum().sort_index()
pelny_okres = pd.date_range(dziennie.index.min(), dziennie.index.max(), freq="D")
print("Dni bez obserwacji:", len(pelny_okres.difference(dziennie.index)))
print(dziennie.describe())
ax = dziennie.plot.hist(bins=12, figsize=(7, 4), edgecolor="white")
ax.set(title="Rozkład dziennego przychodu", xlabel="Dzienny przychód [zł]", ylabel="Liczba dni")
plt.tight_layout()
plt.show()
print(f"Mediana dziennego przychodu wynosi {dziennie.median():.2f} zł dla {len(dziennie)} dni.")


## 3. Przychód miesięczny

Różne miesiące mają różną liczbę dni, dlatego obok sum pokazujemy średnią na dzień z obserwacją.


In [ ]:
sprzedaz["miesiac"] = sprzedaz["data"].dt.month
miesiace = sprzedaz.groupby("miesiac").agg(przychod_zl=("przychod", "sum"), dni=("data", "nunique"))
miesiace["srednio_na_dzien_zl"] = miesiace["przychod_zl"] / miesiace["dni"]
print(miesiace.round(2))
ax = miesiace["srednio_na_dzien_zl"].plot.line(marker="o", figsize=(7, 4))
ax.set(title="Średni przychód na obserwowany dzień", xlabel="Miesiąc 2026", ylabel="Przychód [zł/dzień]", xticks=miesiace.index)
plt.tight_layout()
plt.show()
naj = miesiace["srednio_na_dzien_zl"].idxmax()
print(f"Najwyższa średnia na dzień jest w miesiącu {naj}: {miesiace.loc[naj, 'srednio_na_dzien_zl']:.2f} zł.")


## Ograniczenia i kolejny krok

1. Dane są syntetyczne i obejmują tylko trzy miesiące. Nie uzasadniają prognozy ani twierdzeń o sezonowości prawdziwej kawiarni.
2. Cztery odrzucone pozycje oraz duplikaty wpływają na zakres. Wyniki opisują zachowane pozycje, nie pełny przychód z niekompletnego eksportu.
3. Nie ma kosztów, wspólnych numerów paragonów ani liczby klientów. Nie obliczymy zysku ani przeciętnego wydatku klienta.

Kolejnym krokiem byłoby uzyskanie pełnego eksportu i kosztów produktów. Zanim podejmiemy decyzję biznesową, sprawdzilibyśmy dane źródłowe.
